# LeetCode 1233 — Remove Sub-Folders from the Filesystem bằng Python

Link đề: https://leetcode.com/problems/remove-sub-folders-from-the-filesystem/ · Độ khó: **Medium** · Chủ đề: Array, String, Depth-First Search, Trie

Skeleton LeetCode:

```python
class Solution:
    def removeSubfolders(self, folder: List[str]) -> List[str]:
        ...
```

Ghi chú: trả về các thư mục **không phải** thư mục con của thư mục nào khác trong danh sách; **thứ tự tuỳ ý** (khi kiểm tra ta sắp xếp trước khi so sánh).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Định nghĩa chính xác "thư mục con": `B` là con của `A` khi `B` bắt đầu bằng `A` **theo sau là `'/'`** — không chỉ là "tiền tố chuỗi".
2. Đi từ **so từng cặp** `O(n²·L)` → **tra mọi tiền tố bằng `set`** `O(n·L²)` → **sắp xếp + so với thư mục cha gần nhất** `O(n·L·log n)` → **cây tiền tố (Trie)** theo từng tên thư mục `O(n·L)`.
3. Hiểu vì sao sau khi sắp xếp từ điển, mọi thư mục con của `A` **nằm liền ngay sau** `A` (vì `'/'` nhỏ hơn mọi chữ cái).
4. Tự cài đặt **Trie** bằng `dict` lồng nhau, đánh dấu nút kết thúc và cắt tỉa khi duyệt.
5. Tránh các lỗi: `startswith` không có `'/'` (`"/a/b/ca"` không phải con của `"/a/b/c"`), so với phần tử **ngay trước** thay vì thư mục giữ lại gần nhất, quên sắp xếp.

## 2. Đề bài & ví dụ

Cho danh sách thư mục `folder`, trả về các thư mục sau khi **xoá mọi thư mục con**. Thứ tự kết quả tuỳ ý.

Nếu `folder[i]` nằm bên trong `folder[j]` thì `folder[i]` là **thư mục con** của `folder[j]`. Thư mục con của `folder[j]` phải **bắt đầu bằng `folder[j]`, theo sau là `"/"`**. Ví dụ `"/a/b"` là thư mục con của `"/a"`, nhưng `"/b"` không phải thư mục con của `"/a/b/c"`.

Định dạng đường dẫn: một hoặc nhiều chuỗi nối nhau dạng `'/'` + một hoặc nhiều chữ cái thường. Ví dụ `"/leetcode"` và `"/leetcode/problems"` hợp lệ; chuỗi rỗng và `"/"` không hợp lệ.

**Ví dụ 1:** `["/a","/a/b","/c/d","/c/d/e","/c/f"]` → `["/a","/c/d","/c/f"]`.

**Ví dụ 2:** `["/a","/a/b/c","/a/b/d"]` → `["/a"]`.

**Ví dụ 3:** `["/a/b/c","/a/b/ca","/a/b/d"]` → `["/a/b/c","/a/b/ca","/a/b/d"]`.

**Ràng buộc:**
- `1 <= folder.length <= 4 * 10^4`
- `2 <= folder[i].length <= 100`
- `folder[i]` chỉ gồm chữ cái thường và `'/'`, luôn bắt đầu bằng `'/'`.
- Mỗi tên thư mục là **duy nhất**.

In [1]:
import random
import string
import time
from typing import List

examples = [
    (["/a", "/a/b", "/c/d", "/c/d/e", "/c/f"], ["/a", "/c/d", "/c/f"]),
    (["/a", "/a/b/c", "/a/b/d"], ["/a"]),
    (["/a/b/c", "/a/b/ca", "/a/b/d"], ["/a/b/c", "/a/b/ca", "/a/b/d"]),
]
for f, want in examples:
    print(f, "->", want)

['/a', '/a/b', '/c/d', '/c/d/e', '/c/f'] -> ['/a', '/c/d', '/c/f']
['/a', '/a/b/c', '/a/b/d'] -> ['/a']
['/a/b/c', '/a/b/ca', '/a/b/d'] -> ['/a/b/c', '/a/b/ca', '/a/b/d']


## 3. Kiến thức nền

### 3.1 `str.startswith` và cái bẫy "tiền tố chuỗi"

`s.startswith(p)` kiểm tra `p` có phải tiền tố của `s`. Nhưng `"/a/b/ca".startswith("/a/b/c")` là `True` dù `"ca"` và `"c"` là **hai thư mục khác nhau**. Phải kiểm tra `s.startswith(p + "/")`.

In [2]:
print("/a/b/ca".startswith("/a/b/c"), "<- tiền tố chuỗi, nhưng KHÔNG phải thư mục con")
print("/a/b/ca".startswith("/a/b/c" + "/"), "<- kiểm tra đúng")
print("/a/b/c/x".startswith("/a/b/c" + "/"), "<- thư mục con thật")

True <- tiền tố chuỗi, nhưng KHÔNG phải thư mục con
False <- kiểm tra đúng
True <- thư mục con thật


### 3.2 `str.split("/")` — tách thành tên các cấp

`"/a/b/c".split("/")` cho `["", "a", "b", "c"]` (phần tử đầu rỗng vì chuỗi bắt đầu bằng `/`). Dùng `[1:]` để bỏ phần rỗng. Danh sách tên cấp giúp so sánh **theo đơn vị thư mục**, tránh bẫy ở 3.1.

In [3]:
print("/a/b/c".split("/"))
print("/a/b/c".split("/")[1:], "vs", "/a/b/ca".split("/")[1:])

['', 'a', 'b', 'c']
['a', 'b', 'c'] vs ['a', 'b', 'ca']


### 3.3 Sắp xếp chuỗi theo thứ tự từ điển — vị trí của `'/'`

Python so chuỗi theo mã ký tự. `ord('/') = 47` **nhỏ hơn** mọi chữ cái thường (`ord('a') = 97`). Hệ quả: `"/a/..."` luôn đứng **trước** `"/aa"`, `"/ab"`... nên mọi thư mục con của `"/a"` nằm liền sau `"/a"` trong danh sách đã sắp xếp.

In [4]:
print(ord("/"), ord("a"), ord("z"))
print(sorted(["/ab", "/a/z", "/a", "/aa", "/a/b/c", "/b"]))

47 97 122
['/a', '/a/b/c', '/a/z', '/aa', '/ab', '/b']


### 3.4 `set` và Trie bằng `dict` lồng nhau

- `set`: tra `x in s` trung bình `O(độ dài x)` (phải băm chuỗi).
- **Trie** (cây tiền tố): mỗi nút là một `dict` ánh xạ "tên cấp con" → nút con. Đánh dấu nút là "kết thúc một thư mục" bằng một khoá đặc biệt, ví dụ `"#"` (an toàn vì tên chỉ gồm chữ cái).

In [5]:
trie = {}
for path in ["/a", "/a/b", "/c/d"]:
    node = trie
    for name in path.split("/")[1:]:
        node = node.setdefault(name, {})
    node["#"] = True
print(trie)

{'a': {'#': True, 'b': {'#': True}}, 'c': {'d': {'#': True}}}


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1 bằng cách vẽ cây thư mục:

```
/
├── a          (có trong danh sách)  -> GIỮ
│   └── b      (có)                  -> xoá vì cha "a" có trong danh sách
└── c
    ├── d      (có)                  -> GIỮ (cha "c" không có trong danh sách)
    │   └── e  (có)                  -> xoá vì "d" có
    └── f      (có)                  -> GIỮ
```

Quy tắc: một thư mục được **giữ** khi và chỉ khi **không có tổ tiên nào** của nó nằm trong danh sách. Ví dụ 3: `"/a/b"` không có trong danh sách, nên ba thư mục `c`, `ca`, `d` đều được giữ — `ca` không phải con của `c`.

In [6]:
def ancestors(path):
    parts = path.split("/")[1:]
    return ["/" + "/".join(parts[:k]) for k in range(1, len(parts))]


for p in ["/a/b", "/c/d/e", "/a/b/ca"]:
    print(p, "tổ tiên:", ancestors(p))

/a/b tổ tiên: ['/a']
/c/d/e tổ tiên: ['/c', '/c/d']
/a/b/ca tổ tiên: ['/a', '/a/b']


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi thư mục `f`, kiểm tra **mọi** thư mục `g` khác: nếu `f.startswith(g + "/")` thì `f` là con → bỏ.

In [7]:
def remove_brute(folder):
    ans = []
    for f in folder:
        if not any(f.startswith(g + "/") for g in folder if g != f):
            ans.append(f)
    return ans


for f, want in examples:
    got = remove_brute(f)
    print(sorted(got))
    assert sorted(got) == sorted(want)

['/a', '/c/d', '/c/f']
['/a']
['/a/b/c', '/a/b/ca', '/a/b/d']


**Độ phức tạp:** Time `O(n² · L)` với `n` thư mục, `L ≤ 100` ký tự — `n = 4·10^4` cho `1.6·10^9 · L` phép so sánh: quá chậm. Space `O(1)` ngoài kết quả.

**Nút thắt:** với mỗi `f`, các ứng viên làm "cha" của `f` chỉ là **các tổ tiên** của `f` — tối đa `L/2 ≈ 50` chuỗi — nhưng ta lại đi so với cả `n` chuỗi.

## 6. Bước đột phá

Có ba góc nhìn, mỗi góc dẫn tới một lời giải nhanh:

1. **Chỉ tổ tiên mới quan trọng:** đưa mọi thư mục vào `set`; với `f`, sinh từng tổ tiên (cắt tại mỗi `'/'`) và tra `set`. **Nếu** một tổ tiên có trong `set` **thì** bỏ `f`.
2. **Sắp xếp làm các con "dính" vào cha:** sau khi sắp xếp, duyệt và giữ `last` = thư mục **được giữ** gần nhất. **Nếu** `f.startswith(last + "/")` **thì** `f` là con của `last` → bỏ; **ngược lại** giữ `f` và đặt `last = f`.
   - **Bất biến:** mọi thư mục đã duyệt mà là con của một thư mục trong danh sách đều đã bị bỏ, và `last` là thư mục giữ lại có thứ tự lớn nhất.
3. **Trie theo tên cấp:** chèn mọi thư mục; duyệt Trie từ gốc, **gặp nút đánh dấu kết thúc thì ghi nhận và không đi sâu nữa** (mọi thứ bên dưới là con).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `set` + kiểm tra mọi tổ tiên

In [8]:
def remove_set(folder):
    s = set(folder)
    ans = []
    for f in folder:
        is_sub = False
        for i in range(1, len(f)):
            if f[i] == "/" and f[:i] in s:      # f[:i] là một tổ tiên
                is_sub = True
                break
        if not is_sub:
            ans.append(f)
    return ans


for f, want in examples:
    assert sorted(remove_set(f)) == sorted(want)
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


**Độ phức tạp:** mỗi `f` có tối đa `L/2` tổ tiên, mỗi lần cắt + băm tốn `O(L)` → Time `O(n · L²)` ≈ `4·10^4 · 10^4 = 4·10^8` trong trường hợp xấu nhất lý thuyết, thực tế nhanh hơn nhiều vì đường dẫn thường nông. Space `O(n · L)`.

### 7.2 Cách 3 — Sắp xếp + so với thư mục giữ lại gần nhất

In [9]:
def remove_sort(folder):
    ans = []
    for f in sorted(folder):
        if not ans or not f.startswith(ans[-1] + "/"):
            ans.append(f)
    return ans


for f, want in examples:
    assert sorted(remove_sort(f)) == sorted(want)
print("Cách 3 đúng trên ví dụ:", remove_sort(examples[0][0]))

Cách 3 đúng trên ví dụ: ['/a', '/c/d', '/c/f']


**Độ phức tạp:** sắp xếp `O(n log n)` phép so sánh, mỗi phép `O(L)` → `O(n · L · log n)`. Vòng duyệt `O(n · L)`. Space `O(n · L)` cho bản sắp xếp.

**Vì sao chỉ cần so với `ans[-1]`?** Xem mục 10. Ý chính: nếu `f` là con của một thư mục `A` được giữ, thì giữa `A` và `f` trong thứ tự sắp xếp chỉ toàn các con của `A` (đã bị bỏ), nên `ans[-1]` chính là `A`.

### 7.3 Cách 4 — Trie theo từng tên thư mục

In [10]:
END = "#"   # an toàn vì tên thư mục chỉ gồm chữ cái thường


def remove_trie(folder):
    root = {}
    for path in folder:
        node = root
        for name in path.split("/")[1:]:
            node = node.setdefault(name, {})
        node[END] = path                      # lưu luôn đường dẫn gốc ở nút kết thúc

    ans = []
    stack = [root]
    while stack:                              # DFS lặp
        node = stack.pop()
        if END in node:
            ans.append(node[END])             # ghi nhận và KHÔNG đi sâu nữa
            continue
        for name, child in node.items():
            stack.append(child)
    return ans


for f, want in examples:
    assert sorted(remove_trie(f)) == sorted(want)
print("Cách 4 đúng trên ví dụ:", sorted(remove_trie(examples[0][0])))

Cách 4 đúng trên ví dụ: ['/a', '/c/d', '/c/f']


**Độ phức tạp:** chèn `O(n · L)` (split + duyệt dict), duyệt Trie `O(số nút) ≤ O(n · L)`. Time `O(n · L)`, Space `O(n · L)`.

### 7.4 Cách 5 — Trie, nhưng kiểm tra trong lúc đi xuống

Biến thể hay gặp trong phỏng vấn: chèn hết vào Trie, rồi với **mỗi** thư mục đi từ gốc theo các tên cấp; **nếu** gặp một nút kết thúc **trước** khi tới nút cuối **thì** nó là thư mục con.

In [11]:
def remove_trie_check(folder):
    root = {}
    split = [p.split("/")[1:] for p in folder]
    for parts in split:
        node = root
        for name in parts:
            node = node.setdefault(name, {})
        node[END] = True
    ans = []
    for path, parts in zip(folder, split):
        node = root
        is_sub = False
        for name in parts[:-1]:                # chỉ xét tổ tiên (bỏ cấp cuối)
            node = node[name]
            if END in node:
                is_sub = True
                break
        if not is_sub:
            ans.append(path)
    return ans


for f, want in examples:
    assert sorted(remove_trie_check(f)) == sorted(want)
print("Cách 5 đúng trên ví dụ")

Cách 5 đúng trên ví dụ


**Độ phức tạp:** Time `O(n · L)`, Space `O(n · L)`. Ưu điểm: giữ nguyên **thứ tự** ban đầu của các thư mục được giữ.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1. Sau khi sắp xếp: `["/a", "/a/b", "/c/d", "/c/d/e", "/c/f"]`.

| Bước | f | last (ans[-1]) | f.startswith(last + "/")? | Hành động | ans |
|---|---|---|---|---|---|
| 1 | /a | — | — | giữ | [/a] |
| 2 | /a/b | /a | có (`/a/`) | bỏ | [/a] |
| 3 | /c/d | /a | không | giữ | [/a, /c/d] |
| 4 | /c/d/e | /c/d | có (`/c/d/`) | bỏ | [/a, /c/d] |
| 5 | /c/f | /c/d | không | giữ | [/a, /c/d, /c/f] |

In [12]:
def trace_sort(folder):
    ans = []
    for step, f in enumerate(sorted(folder), 1):
        last = ans[-1] if ans else None
        sub = last is not None and f.startswith(last + "/")
        if not sub:
            ans.append(f)
        print(f"bước {step}: f={f:<9} last={str(last):<8} con của last? {str(sub):<5} -> {'bỏ' if sub else 'giữ'} | ans={ans}")
    return ans


trace_sort(examples[0][0])
print()
trace_sort(examples[2][0])

bước 1: f=/a        last=None     con của last? False -> giữ | ans=['/a']
bước 2: f=/a/b      last=/a       con của last? True  -> bỏ | ans=['/a']
bước 3: f=/c/d      last=/a       con của last? False -> giữ | ans=['/a', '/c/d']
bước 4: f=/c/d/e    last=/c/d     con của last? True  -> bỏ | ans=['/a', '/c/d']
bước 5: f=/c/f      last=/c/d     con của last? False -> giữ | ans=['/a', '/c/d', '/c/f']

bước 1: f=/a/b/c    last=None     con của last? False -> giữ | ans=['/a/b/c']
bước 2: f=/a/b/ca   last=/a/b/c   con của last? False -> giữ | ans=['/a/b/c', '/a/b/ca']
bước 3: f=/a/b/d    last=/a/b/ca  con của last? False -> giữ | ans=['/a/b/c', '/a/b/ca', '/a/b/d']


['/a/b/c', '/a/b/ca', '/a/b/d']

Trace Cách 4 — in cấu trúc Trie và thứ tự DFS:

In [13]:
def trace_trie(folder):
    root = {}
    for path in folder:
        node = root
        for name in path.split("/")[1:]:
            node = node.setdefault(name, {})
        node[END] = path

    def show(node, depth):
        for name, child in sorted((k, v) for k, v in node.items() if k != END):
            mark = f"  <- kết thúc '{child[END]}'" if END in child else ""
            print("  " * depth + name + mark)
            show(child, depth + 1)

    show(root, 0)
    ans, stack = [], [("", root)]
    while stack:
        prefix, node = stack.pop()
        if END in node:
            print(f"DFS gặp kết thúc {node[END]!r}: ghi nhận, cắt nhánh")
            ans.append(node[END])
            continue
        for name, child in sorted(node.items(), reverse=True):
            stack.append((prefix + "/" + name, child))
    return ans


print(trace_trie(examples[0][0]))

a  <- kết thúc '/a'
  b  <- kết thúc '/a/b'
c
  d  <- kết thúc '/c/d'
    e  <- kết thúc '/c/d/e'
  f  <- kết thúc '/c/f'
DFS gặp kết thúc '/a': ghi nhận, cắt nhánh
DFS gặp kết thúc '/c/d': ghi nhận, cắt nhánh
DFS gặp kết thúc '/c/f': ghi nhận, cắt nhánh
['/a', '/c/d', '/c/f']


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def removeSubfolders(self, folder: List[str]) -> List[str]:
        folder.sort()
        ans = [folder[0]]
        for f in folder[1:]:
            last = ans[-1]
            # f là con của last  <=>  f bắt đầu bằng last và ký tự kế tiếp là '/'
            if not (len(f) > len(last) and f.startswith(last) and f[len(last)] == "/"):
                ans.append(f)
        return ans


sol = Solution()
for f, want in examples:
    got = sol.removeSubfolders(list(f))
    print(got)
    assert sorted(got) == sorted(want)

['/a', '/c/d', '/c/f']
['/a']
['/a/b/c', '/a/b/ca', '/a/b/d']


Ghi chú: điều kiện `f.startswith(last) and f[len(last)] == "/"` tương đương `f.startswith(last + "/")` nhưng không tạo chuỗi mới `last + "/"` ở mỗi bước.

## 10. Vì sao đúng (Cách 3)

Gọi danh sách đã sắp xếp là `F[0] < F[1] < ... < F[n−1]`.

**Bổ đề 1:** nếu `B` là con của `A` thì `A < B` (A là tiền tố thật sự của B).

**Bổ đề 2:** nếu `A < C < B` và `B` là con của `A` thì `C` cũng bắt đầu bằng `A + "/"` (tức `C` là con của `A`). Lý do: `A + "/"` là tiền tố của `B`; mọi chuỗi nằm giữa `A` và `B` theo thứ tự từ điển phải có tiền tố `A`; nếu `C = A` thì vô lý (tên duy nhất); nếu ký tự sau `A` trong `C` là chữ cái thì `C > A + "/" + ...` = `B` vì `'/' <` mọi chữ cái — vô lý. Vậy ký tự đó là `'/'`.

**Bất biến:** sau khi xử lý `F[0..k]`, `ans` chứa đúng những thư mục trong `F[0..k]` không có tổ tiên trong danh sách, và `ans[-1]` là phần tử lớn nhất trong `ans`.

- Xét `f = F[k+1]`. Nếu `f` có tổ tiên trong danh sách, gọi `A` là tổ tiên **nông nhất** (gần gốc nhất) — `A` không có tổ tiên nên `A ∈ ans`. Theo Bổ đề 2, mọi phần tử giữa `A` và `f` đều là con của `A` nên **không** nằm trong `ans`. Vậy `ans[-1] = A` và kiểm tra `f.startswith(A + "/")` đúng → bỏ `f`. ✔
- Nếu `f` không có tổ tiên trong danh sách thì `ans[-1]` (một thư mục trong danh sách) không thể là tổ tiên của `f` → giữ `f`. ✔

## 11. Time / Space Complexity

Gọi `n` = số thư mục (≤ `4·10^4`), `L` = độ dài tối đa (≤ 100).

| Cách | Time | Space |
|---|---|---|
| 1. So từng cặp | `O(n² · L)` | `O(1)` |
| 2. Set + tổ tiên | `O(n · L²)` | `O(n · L)` |
| 3. Sort + last | `O(n · L · log n)` | `O(n · L)` |
| 4. Trie + DFS cắt nhánh | `O(n · L)` | `O(n · L)` |
| 5. Trie + kiểm tra khi đi xuống | `O(n · L)` | `O(n · L)` |

Trong Python, Cách 3 thường nhanh nhất thực tế vì `sort` và `startswith` chạy bằng C.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [15]:
methods = {
    "brute": remove_brute,
    "set": remove_set,
    "sort": remove_sort,
    "trie": remove_trie,
    "trie_check": remove_trie_check,
    "submit": lambda f: Solution().removeSubfolders(list(f)),
}
edge = [
    (["/a"], ["/a"]),
    (["/a/b", "/a"], ["/a"]),                                   # cha đứng sau con trong input
    (["/a", "/aa", "/a/a"], ["/a", "/aa"]),                     # "/aa" không phải con của "/a"
    (["/abc/de", "/abc/def", "/abc/de/f"], ["/abc/de", "/abc/def"]),
    (["/a/b/c/d", "/a/b", "/x"], ["/a/b", "/x"]),
    (["/a", "/a/b", "/a/c", "/a/b/c"], ["/a"]),
]
for name, f in methods.items():
    for fol, want in examples + edge:
        assert sorted(f(fol)) == sorted(want), (name, fol)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh đường dẫn từ bảng chữ cái nhỏ `{a, b}` và tên dài 1–2 ký tự để có nhiều ca bẫy như `"/a"` vs `"/aa"`.

In [16]:
def random_path(rng, max_depth=4):
    depth = rng.randint(1, max_depth)
    return "".join("/" + "".join(rng.choice("ab") for _ in range(rng.randint(1, 2))) for _ in range(depth))


rng = random.Random(1233)
for _ in range(400):
    fol = list({random_path(rng) for _ in range(rng.randint(1, 15))})
    rng.shuffle(fol)
    want = sorted(remove_brute(fol))
    for name, f in methods.items():
        assert sorted(f(fol)) == want, (name, fol)
print("400 test ngẫu nhiên: mọi cách khớp brute force")

400 test ngẫu nhiên: mọi cách khớp brute force


### 12.3 Test giới hạn: `4·10^4` thư mục, độ dài tới 100

In [17]:
def random_long_path(rng):
    parts, length = [], 0
    while True:
        name = "".join(rng.choice("abc") for _ in range(rng.randint(1, 6)))
        if length + 1 + len(name) > 100:
            break
        parts.append(name)
        length += 1 + len(name)
        if rng.random() < 0.15:
            break
    return "/" + "/".join(parts)


big = list({random_long_path(rng) for _ in range(45000)})[:40000]
print("số thư mục:", len(big), "| dài nhất:", max(map(len, big)))
results = {}
for name in ["set", "sort", "trie", "trie_check", "submit"]:
    t0 = time.perf_counter()
    results[name] = sorted(methods[name](big))
    print(f"{name:10s} giữ {len(results[name])} thư mục trong {(time.perf_counter() - t0) * 1000:.0f} ms")
assert all(r == results["sort"] for r in results.values())

số thư mục: 38226 | dài nhất: 100
set        giữ 2282 thư mục trong 23 ms
sort       giữ 2282 thư mục trong 19 ms


trie       giữ 2282 thư mục trong 179 ms
trie_check giữ 2282 thư mục trong 207 ms
submit     giữ 2282 thư mục trong 17 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `startswith` không kèm `'/'`

In [18]:
def wrong_no_slash(folder):
    ans = []
    for f in sorted(folder):
        if not ans or not f.startswith(ans[-1]):
            ans.append(f)
    return ans


f, want = examples[2]
print("sai :", wrong_no_slash(f))
print("đúng:", sorted(want))

sai : ['/a/b/c', '/a/b/d']
đúng: ['/a/b/c', '/a/b/ca', '/a/b/d']


**Vì sao sai:** `"/a/b/ca"` có tiền tố chuỗi `"/a/b/c"` nhưng `"ca"` là thư mục khác hẳn `"c"`. Đề nói rõ: thư mục con phải bắt đầu bằng cha **theo sau là `"/"`**. **Sửa:** `f.startswith(last + "/")`.

### Lỗi 2 — So với phần tử **ngay trước** trong danh sách đã sắp xếp

In [19]:
def wrong_prev_element(folder):
    fs = sorted(folder)
    ans = [fs[0]]
    for k in range(1, len(fs)):
        if not fs[k].startswith(fs[k - 1] + "/"):     # so với phần tử trước, không phải thư mục giữ lại
            ans.append(fs[k])
    return ans


f = ["/a", "/a/b", "/a/c"]
print("sai :", wrong_prev_element(f))
print("đúng:", remove_sort(f))

sai : ['/a', '/a/c']
đúng: ['/a']


**Vì sao sai:** `"/a/c"` được so với `"/a/b"` (đã bị bỏ) chứ không phải với `"/a"` → bị giữ nhầm. **Sửa:** so với `ans[-1]` — thư mục **được giữ** gần nhất.

### Lỗi 3 — Quên sắp xếp

In [20]:
def wrong_no_sort(folder):
    ans = []
    for f in folder:
        if not ans or not f.startswith(ans[-1] + "/"):
            ans.append(f)
    return ans


f = ["/a/b", "/c", "/a"]
print("sai :", wrong_no_sort(f))
print("đúng:", remove_sort(f))

sai : ['/a/b', '/c', '/a']
đúng: ['/a', '/c']


**Vì sao sai:** con có thể đứng **trước** cha trong input (`"/a/b"` trước `"/a"`), hoặc bị một thư mục khác chen giữa. Lập luận "con dính liền sau cha" chỉ đúng khi đã sắp xếp. **Sửa:** `sorted(folder)`.

### Lỗi 4 — Trie theo **ký tự** thay vì theo **tên cấp**

In [21]:
def wrong_char_trie(folder):
    root = {}
    for p in folder:
        node = root
        for ch in p:
            node = node.setdefault(ch, {})
        node[END] = p
    ans, stack = [], [root]
    while stack:
        node = stack.pop()
        if END in node:
            ans.append(node[END])
            continue
        stack.extend(v for k, v in node.items())
    return ans


f = ["/a", "/aa"]
print("sai :", sorted(wrong_char_trie(f)))
print("đúng:", sorted(remove_trie(f)))

sai : ['/a']
đúng: ['/a', '/aa']


**Vì sao sai:** Trie theo ký tự coi `"/aa"` là "con" của `"/a"` (lại đúng bẫy tiền tố chuỗi). **Sửa:** Trie theo **tên cấp** (`split("/")`), hoặc nếu dùng ký tự thì chỉ cắt nhánh khi nút kết thúc **và** ký tự kế tiếp là `'/'`.

## 14. Mẹo & nhận dạng pattern

- Bài **đường dẫn / tiền tố** → nghĩ tới **sắp xếp từ điển** (tiền tố đứng trước, các chuỗi cùng tiền tố đứng liền nhau) hoặc **Trie**.
- Luôn hỏi: "tiền tố theo **ký tự** hay theo **đơn vị** (tên thư mục, từ)?" — gần như mọi bug nằm ở đây.
- Trong danh sách đã sắp xếp, so với **phần tử đại diện được giữ** (`ans[-1]`), không phải phần tử liền trước.
- Muốn giữ thứ tự gốc → dùng `set`/Trie kiểm tra từng phần tử thay vì sắp xếp.
- Ký tự phân cách nhỏ hơn mọi ký tự tên (`'/' < 'a'`) là lý do sắp xếp hoạt động; với bảng chữ khác hãy kiểm tra lại giả thiết này.

## 15. Biến thể

1. **Trả về các thư mục "lá"** (không có con nào trong danh sách) — ngược lại bài này.
2. **Đếm số thư mục con** của mỗi thư mục được giữ.
3. **Đường dẫn có `.` và `..`** → phải chuẩn hoá trước (Simplify Path — LeetCode 71).
4. **Delete Duplicate Folders in System (LeetCode 1948, Hard):** Trie + băm cấu trúc cây con.
5. **Replace Words (LeetCode 648):** thay mỗi từ bằng **gốc từ** ngắn nhất là tiền tố — cùng ý tưởng Trie dừng ở nút kết thúc đầu tiên.

In [22]:
def leaf_folders(folder):
    """Biến thể 1: thư mục không có thư mục con nào trong danh sách."""
    fs = sorted(folder)
    return [fs[k] for k in range(len(fs)) if not (k + 1 < len(fs) and fs[k + 1].startswith(fs[k] + "/"))]


for f, _ in examples:
    print(f, "-> lá:", leaf_folders(f))

['/a', '/a/b', '/c/d', '/c/d/e', '/c/f'] -> lá: ['/a/b', '/c/d/e', '/c/f']
['/a', '/a/b/c', '/a/b/d'] -> lá: ['/a/b/c', '/a/b/d']
['/a/b/c', '/a/b/ca', '/a/b/d'] -> lá: ['/a/b/c', '/a/b/ca', '/a/b/d']


Ghi chú biến thể 1: sau khi sắp xếp, nếu `fs[k]` có con thì con **nhỏ nhất** của nó nằm ngay ở `fs[k+1]` (Bổ đề 2), nên chỉ cần nhìn phần tử kế tiếp.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | `startswith(g + "/")` với mọi `g` | `O(n²·L)` | `O(1)` | trọng tài |
| 2. Set + tổ tiên | tra mọi tiền tố tại `'/'` | `O(n·L²)` | `O(n·L)` | giữ thứ tự gốc, code ngắn |
| 3. Sort + last | con dính liền sau cha | `O(n·L·log n)` | `O(n·L)` | **nộp bài**, nhanh trong Python |
| 4. Trie + DFS | dừng ở nút kết thúc đầu tiên | `O(n·L)` | `O(n·L)` | khi đã quen Trie, nhiều truy vấn |
| 5. Trie + kiểm tra | đi xuống, gặp kết thúc sớm → con | `O(n·L)` | `O(n·L)` | giữ thứ tự gốc |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`sorted(Solution().removeSubfolders(["/x/y", "/x", "/xy", "/x/yz/w"]))` = ?

### Bài 2 (tính tay)

Sắp xếp `["/b/a", "/a/b/c", "/b", "/a/bc", "/a/b"]` rồi trace Cách 3.

### Bài 3 (điền TODO) — đếm số thư mục con bị xoá của mỗi thư mục được giữ

In [23]:
def count_removed(folder):
    """Trả về dict: thư mục được giữ -> số thư mục con (trong danh sách) bị xoá vì nó."""
    res = {}
    last = None
    for f in sorted(folder):
        # TODO: nếu f là con của last thì res[last] += 1, ngược lại giữ f (res[f] = 0, last = f)
        raise NotImplementedError
    return res

# print(count_removed(["/a", "/a/b", "/c/d", "/c/d/e", "/c/f", "/a/b/x"]))   # {'/a': 2, '/c/d': 1, '/c/f': 0}

### Bài 4 (tự viết lại)

Viết lại Cách 2 nhưng dùng `split("/")` và ghép dần tổ tiên `"/" + "/".join(parts[:k])` thay vì cắt chuỗi ở vị trí `'/'`.

In [24]:
def remove_set_split_todo(folder):
    # TODO
    raise NotImplementedError

# print(sorted(remove_set_split_todo(["/a", "/a/b", "/c/d", "/c/d/e", "/c/f"])))   # ['/a', '/c/d', '/c/f']

### Bài 5 (mini challenge) — Replace Words

Cho `roots` (danh sách gốc từ) và câu `sentence`, thay mỗi từ bằng gốc từ **ngắn nhất** là tiền tố của nó (nếu có). Dùng Trie theo ký tự (ở đây tiền tố ký tự là đúng ý đề!).

In [25]:
def replace_words_todo(roots, sentence):
    # TODO
    raise NotImplementedError

# print(replace_words_todo(["cat", "bat", "rat"], "the cattle was rattled by the battery"))   # "the cat was rat by the bat"

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: sắp xếp: `["/x", "/x/y", "/x/yz/w", "/xy"]`. Mức 2: `"/x/y"` và `"/x/yz/w"` bắt đầu bằng `"/x/"`. Mức 3: `["/x", "/xy"]`.

**Bài 2.** Mức 1: sau sắp xếp `["/a/b", "/a/b/c", "/a/bc", "/b", "/b/a"]`. Mức 2: `"/a/b/c"` là con của `"/a/b"`; `"/a/bc"` thì không. Mức 3: kết quả `["/a/b", "/a/bc", "/b"]`.

**Bài 3.** Mức 1: điều kiện `last is not None and f.startswith(last + "/")`. Mức 2: nhánh đúng: `res[last] += 1`. Mức 3: nhánh sai: `res[f] = 0; last = f`.

**Bài 4.** Mức 1: `parts = f.split("/")[1:]`. Mức 2: tổ tiên ứng với `k = 1..len(parts)−1`. Mức 3: `any(("/" + "/".join(parts[:k])) in s for k in ...)`.

**Bài 5.** Mức 1: chèn mọi gốc từ vào Trie ký tự, đánh dấu kết thúc. Mức 2: với mỗi từ, đi theo từng ký tự; gặp nút kết thúc → trả tiền tố hiện tại. Mức 3: nếu rơi khỏi Trie → giữ nguyên từ.

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
r1 = sorted(Solution().removeSubfolders(["/x/y", "/x", "/xy", "/x/yz/w"]))
print(r1)
assert r1 == ["/x", "/xy"]
r2 = trace_sort(["/b/a", "/a/b/c", "/b", "/a/bc", "/a/b"])
assert r2 == ["/a/b", "/a/bc", "/b"]

['/x', '/xy']
bước 1: f=/a/b      last=None     con của last? False -> giữ | ans=['/a/b']
bước 2: f=/a/b/c    last=/a/b     con của last? True  -> bỏ | ans=['/a/b']
bước 3: f=/a/bc     last=/a/b     con của last? False -> giữ | ans=['/a/b', '/a/bc']
bước 4: f=/b        last=/a/bc    con của last? False -> giữ | ans=['/a/b', '/a/bc', '/b']
bước 5: f=/b/a      last=/b       con của last? True  -> bỏ | ans=['/a/b', '/a/bc', '/b']


In [27]:
def count_removed(folder):
    res, last = {}, None
    for f in sorted(folder):
        if last is not None and f.startswith(last + "/"):
            res[last] += 1
        else:
            res[f] = 0
            last = f
    return res


def remove_set_split(folder):
    s = set(folder)
    ans = []
    for f in folder:
        parts = f.split("/")[1:]
        if not any(("/" + "/".join(parts[:k])) in s for k in range(1, len(parts))):
            ans.append(f)
    return ans


def replace_words(roots, sentence):
    trie = {}
    for r in roots:
        node = trie
        for ch in r:
            node = node.setdefault(ch, {})
        node[END] = True
    out = []
    for w in sentence.split():
        node, rep = trie, w
        for i, ch in enumerate(w):
            if ch not in node:
                break
            node = node[ch]
            if END in node:
                rep = w[:i + 1]
                break
        out.append(rep)
    return " ".join(out)


assert count_removed(["/a", "/a/b", "/c/d", "/c/d/e", "/c/f", "/a/b/x"]) == {"/a": 2, "/c/d": 1, "/c/f": 0}
assert replace_words(["cat", "bat", "rat"], "the cattle was rattled by the battery") == "the cat was rat by the bat"
rng = random.Random(55)
for _ in range(300):
    fol = list({random_path(rng) for _ in range(rng.randint(1, 12))})
    assert sorted(remove_set_split(fol)) == sorted(remove_brute(fol))
    cr = count_removed(fol)
    assert sorted(cr) == sorted(remove_brute(fol))
    for keep, c in cr.items():
        assert c == sum(g.startswith(keep + "/") for g in fol)
    roots = list({"".join(rng.choice("ab") for _ in range(rng.randint(1, 3))) for _ in range(3)})
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 5))) for _ in range(5)]
    expect = " ".join(min((r for r in roots if w.startswith(r)), key=len, default=w) for w in words)
    assert replace_words(roots, " ".join(words)) == expect
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
folder.sort()                              # cha đứng trước, con dính liền sau
ans = []
for f in folder:
    if not ans or not f.startswith(ans[-1] + "/"):   # nhớ dấu '/'
        ans.append(f)
return ans
```

**Checklist:**
- [ ] Kiểm tra `startswith(last + "/")`, không phải `startswith(last)`?
- [ ] So với `ans[-1]` (thư mục **được giữ**), không phải phần tử liền trước?
- [ ] Đã `sort` (con có thể đứng trước cha trong input)?
- [ ] Nếu dùng Trie: theo **tên cấp**, không theo ký tự?
- [ ] Đã thử bẫy `"/a"` vs `"/aa"`, `"/a/b/c"` vs `"/a/b/ca"`?
- [ ] Khi tự test: so sánh sau khi `sorted` (thứ tự tuỳ ý)?